In [ ]:
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from joblib import Parallel, delayed
from tqdm.auto import tqdm
from genkit.plotting import PRETTY_RCPARAMS

repo_root = Path.cwd().resolve()
while repo_root != repo_root.parent and not (repo_root / "src" / "genkit").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from benchmarks.analysis.utils import (
    METRIC_ORDER,
    aggregated_best_alpha,
    best_table_df,
    evaluate_run,
    format_mean_std_latex,
    reduce_best_alpha,
    run_dirs,
    setting_code,
    summarize_exact,
)

plt.rcParams.update(PRETTY_RCPARAMS)


In [ ]:
config_path = repo_root / "benchmarks" / "configs" / "01_benchmark.yaml"
analysis_dir = repo_root / "benchmarks" / "analysis"
data_dir = analysis_dir / "data"
data_dir.mkdir(parents=True, exist_ok=True)
batch_dirs = sorted(path for path in data_dir.iterdir() if path.is_dir())
if not batch_dirs:
    raise FileNotFoundError(f"No benchmark batch found under {data_dir}")
batch_dir = batch_dirs[-1]
cache_path = batch_dir / "_metrics_cache_n4096_r3.csv"

device = "cuda" if torch.cuda.is_available() else "cpu"
n_eval_samples = 4096
n_repeats = 3
n_jobs = 1 if device == "cuda" else 4
metric_order = METRIC_ORDER

print(f"device={device} | batch_dir={batch_dir} | n_runs={len(run_dirs(batch_dir))}")

manifest_paths = sorted(batch_dir.glob("manifest_shard_*.csv"))
summary_paths = sorted(batch_dir.glob("summary_shard_*.txt"))
shards_present = sorted(int(path.stem.split("_")[-1]) for path in manifest_paths)
if shards_present:
    all_shards = list(range(max(shards_present) + 1))
    missing_shards = [idx for idx in all_shards if idx not in shards_present]
else:
    missing_shards = []
print(f"manifests={len(manifest_paths)} summaries={len(summary_paths)} shards_present={shards_present} missing_shards={missing_shards}")


In [ ]:
if cache_path.exists():
    repeat_df = pd.read_csv(cache_path)
else:
    paths = [run_dir for run_dir in run_dirs(batch_dir) if (run_dir / "checkpoint.pt").exists()]
    rows_nested = Parallel(n_jobs=n_jobs, backend="loky")(
        delayed(evaluate_run)(run_dir, n_samples=n_eval_samples, n_repeats=n_repeats, device=device)
        for run_dir in tqdm(paths, desc="Evaluating runs")
    )
    repeat_df = pd.DataFrame([row for rows in rows_nested for row in rows])
    repeat_df.to_csv(cache_path, index=False)

repeat_df = repeat_df[repeat_df["model_label"] != "ScoreSDE-Orig"].copy()
repeat_df["model_alpha"] = pd.to_numeric(repeat_df["model_alpha"], errors="coerce")
summary_exact = summarize_exact(repeat_df)
summary_exact = summary_exact[summary_exact["model_label"] != "ScoreSDE-Orig"].copy()
summary_exact["model_alpha"] = pd.to_numeric(summary_exact["model_alpha"], errors="coerce")

print(cache_path)
print(repeat_df.shape, summary_exact.shape)

failed_rows = []
for manifest_path in manifest_paths:
    manifest_df = pd.read_csv(manifest_path)
    if "status" in manifest_df.columns:
        failed_rows.append(manifest_df[manifest_df["status"] != "ok"].copy())
failed_df = pd.concat(failed_rows, ignore_index=True) if failed_rows else pd.DataFrame()
print(f"failed_manifest_rows={len(failed_df)}")
if len(failed_df):
    cols = [col for col in ["combo_index", "trial_idx", "model_preset", "train_preset", "status", "error_type"] if col in failed_df.columns]
    print(failed_df[cols].to_string(index=False))


In [ ]:
for metric in metric_order:
    winners = best_table_df(summary_exact, metric)
    table = pd.DataFrame(
        index=sorted(winners["model_label"].unique()),
        columns=sorted(winners["dataset_alpha"].unique()),
        dtype=object,
    )
    for dataset_alpha, group in winners.groupby("dataset_alpha"):
        best_model = group.nsmallest(1, f"{metric}_mean").iloc[0]["model_label"]
        for _, row in group.iterrows():
            value = format_mean_std_latex(
                row[f"{metric}_mean"],
                row[f"{metric}_std"],
                caption=setting_code(row),
                bold=row["model_label"] == best_model,
            )
            table.loc[row["model_label"], dataset_alpha] = value

    fig, ax = plt.subplots(1, 1, figsize=(0.8 + len(table.columns), 0.8 + 0.35 * len(table.index)))
    ax.axis("off")
    mtbl = ax.table(
        cellText=table.values,
        rowLabels=list(table.index),
        colLabels=[rf"$\mathbf{{\alpha={alpha:.3g}}}$" for alpha in table.columns],
        loc="center",
        cellLoc="center",
    )
    mtbl.auto_set_font_size(False)
    mtbl.set_fontsize(8)
    mtbl.scale(1.0, 1.5)
    corner_width = mtbl[(1, -1)].get_width() if (1, -1) in mtbl.get_celld() else mtbl[(1, 0)].get_width()
    corner_height = mtbl[(0, 0)].get_height()
    mtbl.add_cell(0, -1, width=corner_width, height=corner_height, text=metric, loc="center")
    for (row, col), cell in mtbl.get_celld().items():
        cell.set_edgecolor("0.3")
        cell.set_linewidth(0.3)
        if col == -1:
            cell.set_text_props(weight="bold")
    plt.show()


In [ ]:
def grouped_violin(ax, series_by_label, title, ylabel):
    labels = list(series_by_label)
    values = [np.asarray(series_by_label[label], dtype=float) for label in labels]
    pos = np.arange(1, len(labels) + 1)
    violins = ax.violinplot(values, positions=pos, widths=0.8, showmeans=True, showextrema=False)
    for body in violins["bodies"]:
        body.set_alpha(0.35)
    violins["cmeans"].set_color("black")
    for idx, vals in enumerate(values, start=1):
        jitter = np.linspace(-0.05, 0.05, len(vals)) if len(vals) > 1 else np.array([0.0])
        ax.scatter(np.full(len(vals), idx) + jitter, vals, s=12, alpha=0.7, color="black")
    ax.set_xticks(pos, labels)
    ax.tick_params(axis="x", labelrotation=30)
    for tick in ax.get_xticklabels():
        tick.set_ha("right")
    ax.set_title(title)
    ax.set_ylabel(ylabel)
    ax.grid(alpha=0.2, axis="y")

label_maps = {
    "train_preset": {"short": "Short Train", "long": "Long Train"},
    "network_preset": {"mlp_small": "Small Net", "mlp_medium": "Medium Net"},
}

for metric in ["W1", "MSSLE_95", "MSSLE_99"]:
    agg = aggregated_best_alpha(summary_exact, metric)
    groups = [
        ("train_preset", f"{metric}: train"),
        ("network_preset", f"{metric}: network"),
        ("model_label", f"{metric}: model"),
    ]
    fig, axes = plt.subplots(1, 3, figsize=(8.0, 2.0), constrained_layout=True)
    for ax, (column, title) in zip(axes, groups):
        label_map = label_maps.get(column, {})
        series_by_label = {label_map.get(key, key): grp[metric].to_numpy() for key, grp in agg.groupby(column)}
        grouped_violin(ax, series_by_label, title, metric)
    plt.show()


In [ ]:
source_pairs = [
    ("GF-Linear", "ASF-Linear", "Flow baselines"),
    ("DDPM-V", "DLPM-Orig", "Diffusion baselines"),
]

for metric in ["W1", "MSSLE_99"]:
    reduced = reduce_best_alpha(summary_exact, metric)
    best_over_net_train = (
        reduced.sort_values(f"{metric}_mean")
        .groupby(["dataset_alpha", "model_label"], dropna=False)
        .first()
        .reset_index()
    )
    fig, axes = plt.subplots(1, 2, figsize=(6.0, 2.5), constrained_layout=True)
    for ax, (left_model, right_model, title) in zip(axes, source_pairs):
        sub = best_over_net_train[best_over_net_train["model_label"].isin([left_model, right_model])]
        for model_label, group in sub.groupby("model_label"):
            group = group.sort_values("dataset_alpha")
            ax.plot(group["dataset_alpha"], group[f"{metric}_mean"], marker="o", lw=2.5, label=model_label, alpha=0.7)
        ax.set_title(f"{title} | {metric}")
        ax.set_xlabel(r"$\alpha_{\mathrm{data}}$")
        ax.set_ylabel(metric)
        ax.grid(alpha=0.2)
        ax.legend(frameon=False)
    plt.show()


In [ ]:
for model_label in ["ASF-Linear", "DLPM-Orig"]:
    fig, axes = plt.subplots(1, 2, figsize=(5.8, 2.0), constrained_layout=True)
    sub = summary_exact[summary_exact["model_label"] == model_label].copy()
    for ax, metric in zip(axes, ["W1", "MSSLE_99"]):
        best = (
            sub.sort_values(f"{metric}_mean")
            .groupby(["dataset_alpha", "model_alpha"], dropna=False)
            .first()
            .reset_index()
        )
        pivot = best.pivot(index="dataset_alpha", columns="model_alpha", values=f"{metric}_mean").sort_index().sort_index(axis=1)
        image = ax.imshow(pivot.values, origin="lower", aspect="auto", interpolation="bilinear")
        ax.set_title(f"{model_label} | {metric}")
        ax.set_xlabel(r"$\alpha_{\mathrm{model}}$")
        ax.set_ylabel(r"$\alpha_{\mathrm{data}}$")
        ax.set_xticks(np.arange(len(pivot.columns)), [f"{x:.3g}" for x in pivot.columns])
        ax.set_yticks(np.arange(len(pivot.index)), [f"{x:.3g}" for x in pivot.index])
        fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
    plt.show()


In [ ]:
for metric in ["W1", "MSSLE_99"]:
    fig, ax = plt.subplots(figsize=(3.0, 2.5), constrained_layout=True)
    for model_label in ["ASF-Linear", "DLPM-Orig"]:
        sub = summary_exact[summary_exact["model_label"] == model_label].copy()
        sub = (
            sub.sort_values(f"{metric}_mean")
            .groupby(["dataset_alpha", "model_alpha"], dropna=False)
            .first()
            .reset_index()
        )
        sub["alpha_gap"] = np.abs(sub["dataset_alpha"] - sub["model_alpha"])
        curve = sub.groupby("alpha_gap", dropna=False)[f"{metric}_mean"].mean().reset_index().sort_values("alpha_gap")
        ax.plot(curve["alpha_gap"], curve[f"{metric}_mean"], marker="o", lw=2.5, label=model_label, alpha=0.7)
    ax.set_title(f"Alpha mismatch | {metric}")
    ax.set_xlabel(r"$|\alpha_{\mathrm{model}} - \alpha_{\mathrm{data}}|$")
    ax.set_ylabel(metric)
    ax.grid(alpha=0.2)
    ax.legend(frameon=False)
    plt.show()
